# LAB-D2-03: Train a Neural Network with NumPy

**Purpose:** Integrate initialization, vectorized forward propagation, stable loss, backpropagation, mini-batches, updates, metrics, and visual evidence into one trainable network.

**Objectives:** `OBJ-D2-01`, `OBJ-D2-03`, `OBJ-D2-04`, `OBJ-D2-05`, `OBJ-D2-06`  
**Estimated duration:** 70 minutes live; CPU baseline under 60 seconds  
**Prerequisites:** `LAB-D1-04`, `LAB-D2-01`, `LAB-D2-02`, `LESSON-D2-04`, `ACT-D2-03`  
**Environment:** CPU required; NumPy, matplotlib, scikit-learn; 800 generated examples; no network or GPU required

Workflow: **Observe -> Predict -> Implement one stage -> Assert -> Train -> Inspect -> Diagnose -> Run one controlled experiment -> Explain**. Restart and run in order. The first commitment cell intentionally stops execution.

In [ ]:
import copy
import platform
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__} | scikit-learn {sklearn.__version__}")
print("Runtime target: local/Colab CPU; no download or GPU required.")

## Recap: The Training-Loop State Machine

Each mini-batch moves through the same responsibilities:

`initialize -> forward/cache -> loss -> backward -> finite-state check -> update -> repeat -> evaluate`

An epoch contains multiple mini-batch updates. Validation examples may measure the current parameters, but they must not produce updates or preprocessing statistics.

## Local Dataset: Seeded, Stratified Moons

Generate 800 noisy examples, reserve 25% for validation with stratification, and standardize using training-set statistics only. Targets use shape `(B, 1)` because this lab's binary output also has shape `(B, 1)`. The same split specification is reused in `LAB-D2-04`.

In [ ]:
DATA_SEED = 23
MODEL_SEED = 24
BATCH_SEED = 25
X_all, y_all = make_moons(n_samples=800, noise=0.27, random_state=DATA_SEED)
X_train_raw, X_val_raw, y_train_flat, y_val_flat = train_test_split(
    X_all, y_all, test_size=0.25, random_state=DATA_SEED, stratify=y_all
)
train_mean = X_train_raw.mean(axis=0, keepdims=True)
train_std = X_train_raw.std(axis=0, keepdims=True)
X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std
y_train = y_train_flat.reshape(-1, 1).astype(np.float64)
y_val = y_val_flat.reshape(-1, 1).astype(np.float64)
assert X_train.shape == (600, 2) and X_val.shape == (200, 2)
assert y_train.shape == (600, 1) and y_val.shape == (200, 1)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert abs(y_train.mean() - y_val.mean()) < 0.01
print(f"Train: {X_train.shape}, positive rate {y_train.mean():.3f}")
print(f"Validation: {X_val.shape}, positive rate {y_val.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
for ax, features, targets, title in [
    (axes[0], X_train, y_train_flat, 'Training split'),
    (axes[1], X_val, y_val_flat, 'Validation split'),
]:
    for label, marker, color in [(0, 'o', '#277da1'), (1, '^', '#f9844a')]:
        mask = targets == label
        ax.scatter(features[mask, 0], features[mask, 1], marker=marker, color=color, edgecolor='black', linewidth=0.3, alpha=0.72, label=f'class {label}')
    ax.set(title=title, xlabel='standardized x1', ylabel='standardized x2')
    ax.legend()
plt.tight_layout()
plt.show()

## Predict Before Building

For a `2 -> 8 -> 1` network with tanh hidden activation and sigmoid output, predict all parameter/cache shapes, the initial BCE band, the early loss direction, and the boundary family. Then predict how removing tanh differs from merely choosing a poor learning rate.

In [ ]:
build_predictions = {
    "W1_b1_W2_b2_shapes": "",
    "Z1_A1_Z2_P_batch_shapes": "",
    "initial_BCE_band": "",
    "early_loss_direction": "",
    "baseline_boundary_family": "",
    "no_activation_vs_bad_rate": "",
}
assert all(value.strip() for value in build_predictions.values()), (
    "Prediction checkpoint: complete every shape, range, and mechanism field before implementation."
)

## Stage 1: Scaled Initialization

Complete `initialize_parameters`. Use the supplied generator and a scale proportional to $\sqrt{2/(d_{in}+d_{out})}$ for each weight matrix, multiplied by `gain`. Biases begin at zero. Return `W1`, `b1`, `W2`, and `b2`.

In [ ]:
def initialize_parameters(input_dim, hidden_dim, output_dim, seed, gain=0.2):
    # TODO: create scaled W1/W2 and zero b1/b2 with the requested shapes.
    raise NotImplementedError("TODO: implement scaled initialization")

baseline_parameters = initialize_parameters(2, 8, 1, MODEL_SEED)
assert baseline_parameters['W1'].shape == (2, 8)
assert baseline_parameters['b1'].shape == (8,)
assert baseline_parameters['W2'].shape == (8, 1)
assert baseline_parameters['b2'].shape == (1,)
assert all(np.all(np.isfinite(values)) for values in baseline_parameters.values())
print("Parameter shapes:", {name: values.shape for name, values in baseline_parameters.items()})

## Stage 2: Vectorized Forward Cache and Stable BCE

Implement stable sigmoid, forward propagation, and clipped BCE. `forward` supports `activation='tanh'` for the baseline and `activation='identity'` for a later controlled failure. Return `X`, `Z1`, `A1`, `Z2`, and `P`.

In [ ]:
def sigmoid(values):
    # TODO: implement a stable elementwise sigmoid.
    raise NotImplementedError("TODO: implement stable sigmoid")

def forward(X, parameters, activation='tanh'):
    # TODO: compute the complete batch-first cache for tanh or identity hidden activation.
    raise NotImplementedError("TODO: implement vectorized forward propagation")

def binary_cross_entropy(y_true, probabilities, epsilon=1e-12):
    # TODO: return mean clipped BCE.
    raise NotImplementedError("TODO: implement clipped BCE")

In [ ]:
initial_cache = forward(X_train, baseline_parameters)
initial_loss = binary_cross_entropy(y_train, initial_cache['P'])
assert initial_cache['Z1'].shape == initial_cache['A1'].shape == (600, 8)
assert initial_cache['Z2'].shape == initial_cache['P'].shape == (600, 1)
assert np.all(np.isfinite(initial_cache['P']))
assert np.all((0.0 <= initial_cache['P']) & (initial_cache['P'] <= 1.0))
assert 0.65 <= initial_loss <= 0.75
print(f"Initial training BCE: {initial_loss:.6f}")
print("Initial cache shapes:", {name: value.shape for name, value in initial_cache.items()})

## Stage 3: Vectorized Backpropagation

Complete `backward`. Begin with sigmoid-plus-BCE output sensitivity, propagate through `W2`, and use either the tanh local derivative or identity derivative. Average over the current mini-batch and return gradient arrays keyed like the parameters.

In [ ]:
def backward(cache, parameters, y_true, activation='tanh'):
    # TODO: compute vectorized W1, b1, W2, and b2 gradients.
    raise NotImplementedError("TODO: implement vectorized backpropagation")

initial_gradients = backward(initial_cache, baseline_parameters, y_train)
assert set(initial_gradients) == set(baseline_parameters)
for name in baseline_parameters:
    assert initial_gradients[name].shape == baseline_parameters[name].shape
    assert np.all(np.isfinite(initial_gradients[name]))
print("Gradient shapes:", {name: value.shape for name, value in initial_gradients.items()})

## Stage 4: Mini-Batches and Non-Mutating Updates

Implement a shuffled mini-batch iterator and parameter update. One epoch must visit every training example exactly once; the final batch may be smaller. Return new parameter arrays rather than mutating the input dictionary.

In [ ]:
def mini_batches(X, y, batch_size, rng):
    # TODO: yield shuffled X_batch, y_batch, and original indices for one epoch.
    raise NotImplementedError("TODO: implement the mini-batch iterator")

def update_parameters(parameters, gradients, learning_rate):
    # TODO: return new arrays after subtracting learning_rate * gradient.
    raise NotImplementedError("TODO: implement parameter updates")

batch_rng = np.random.default_rng(BATCH_SEED)
batch_index_groups = [indices for _, _, indices in mini_batches(X_train, y_train, 64, batch_rng)]
visited = np.concatenate(batch_index_groups)
assert len(batch_index_groups) == 10
assert np.array_equal(np.sort(visited), np.arange(len(X_train)))
assert len(np.unique(visited)) == len(X_train)
assert batch_index_groups[-1].shape == (24,)
print("Mini-batch sizes:", [len(indices) for indices in batch_index_groups])

## Stage 5: Training Loop with Bounded Diagnostics

Complete `train_model`. Each epoch must shuffle mini-batches, run forward/backward/update, check finite parameters and gradients, and record train/validation loss, accuracy, and mean gradient norm. Refuse more than 2,000 epochs so an accidental unstable run cannot scroll indefinitely.

In [ ]:
def classification_accuracy(y_true, probabilities):
    return float(np.mean((probabilities >= 0.5).astype(int) == y_true.astype(int)))

def gradient_norm(gradients):
    return float(np.sqrt(sum(np.sum(values ** 2) for values in gradients.values())))

def require_finite_state(parameters, gradients, epoch, batch_index):
    parameter_ok = all(np.all(np.isfinite(values)) for values in parameters.values())
    gradient_ok = all(np.all(np.isfinite(values)) for values in gradients.values())
    if not parameter_ok or not gradient_ok:
        raise FloatingPointError(
            f"Non-finite state at epoch {epoch}, batch {batch_index}. Stop this run; inspect the rate, loss, and gradient norm."
        )

def train_model(X_train, y_train, X_val, y_val, *, hidden_dim, learning_rate, epochs, batch_size, activation, model_seed, batch_seed):
    # TODO: initialize, train, check finite state, and return parameters plus a history dictionary.
    raise NotImplementedError("TODO: implement the complete training loop")

## Predict the Baseline Evidence

The baseline uses hidden width `8`, learning rate `0.6`, 600 epochs, batch size `64`, and tanh. Predict the final loss/accuracy bands, whether mini-batch loss must be monotonic, and what a healthy gradient-norm strip should rule out.

In [ ]:
baseline_predictions = {
    "final_loss_band": "",
    "train_accuracy_band": "",
    "validation_accuracy_band": "",
    "mini_batch_monotonicity": "",
    "gradient_norm_evidence": "",
}
assert all(value.strip() for value in baseline_predictions.values())

In [ ]:
baseline_config = {
    "hidden_dim": 8, "learning_rate": 0.6, "epochs": 600, "batch_size": 64,
    "activation": 'tanh', "model_seed": MODEL_SEED, "batch_seed": BATCH_SEED,
}
started = time.perf_counter()
trained_parameters, baseline_history = train_model(X_train, y_train, X_val, y_val, **baseline_config)
baseline_seconds = time.perf_counter() - started
baseline_train_cache = forward(X_train, trained_parameters)
baseline_val_cache = forward(X_val, trained_parameters)
baseline_train_loss = binary_cross_entropy(y_train, baseline_train_cache['P'])
baseline_val_loss = binary_cross_entropy(y_val, baseline_val_cache['P'])
baseline_train_accuracy = classification_accuracy(y_train, baseline_train_cache['P'])
baseline_val_accuracy = classification_accuracy(y_val, baseline_val_cache['P'])
print(f"Baseline runtime: {baseline_seconds:.3f}s")
print(f"Train loss/accuracy: {baseline_train_loss:.4f} / {baseline_train_accuracy:.3f}")
print(f"Validation loss/accuracy: {baseline_val_loss:.4f} / {baseline_val_accuracy:.3f}")
assert baseline_seconds < 60.0
assert baseline_train_loss < 0.35
assert 0.85 <= baseline_train_accuracy <= 0.95
assert 0.85 <= baseline_val_accuracy <= 0.95

## Inspect: Baseline Evidence Dashboard

The dashboard separates optimization progress (loss and gradient norm), task performance (accuracy), and representation evidence (decision region). Shared epoch axes make train/validation comparison fair. A nonlinear boundary is visible evidence, not proof that every error is explained.

In [ ]:
def make_mesh(X, padding=0.6, resolution=180):
    x1 = np.linspace(X[:, 0].min() - padding, X[:, 0].max() + padding, resolution)
    x2 = np.linspace(X[:, 1].min() - padding, X[:, 1].max() + padding, resolution)
    xx1, xx2 = np.meshgrid(x1, x2)
    return xx1, xx2, np.column_stack([xx1.ravel(), xx2.ravel()])

def plot_boundary(ax, parameters, activation, title):
    xx1, xx2, mesh = make_mesh(np.vstack([X_train, X_val]))
    mesh_probability = forward(mesh, parameters, activation)['P'].reshape(xx1.shape)
    ax.contourf(xx1, xx2, mesh_probability, levels=np.linspace(0, 1, 11), cmap='RdYlBu_r', alpha=0.45)
    ax.contour(xx1, xx2, mesh_probability, levels=[0.5], colors='black', linewidths=2)
    for label, marker, color in [(0, 'o', '#277da1'), (1, '^', '#f9844a')]:
        mask = y_val_flat == label
        ax.scatter(X_val[mask, 0], X_val[mask, 1], marker=marker, color=color, edgecolor='black', linewidth=0.3, s=30, label=f'class {label}')
    ax.set(title=title, xlabel='standardized x1', ylabel='standardized x2')
    ax.legend()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_axis = np.arange(1, len(baseline_history['train_loss']) + 1)
axes[0, 0].plot(epochs_axis, baseline_history['train_loss'], label='train loss', color='#277da1')
axes[0, 0].plot(epochs_axis, baseline_history['val_loss'], label='validation loss', color='#f9844a')
axes[0, 0].set(title='Loss evidence', xlabel='epoch', ylabel='BCE')
axes[0, 0].legend()
axes[0, 1].plot(epochs_axis, baseline_history['train_accuracy'], label='train accuracy', color='#277da1')
axes[0, 1].plot(epochs_axis, baseline_history['val_accuracy'], label='validation accuracy', color='#f9844a')
axes[0, 1].set(title='Accuracy evidence', xlabel='epoch', ylabel='accuracy', ylim=(0.45, 1.0))
axes[0, 1].legend()
axes[1, 0].plot(epochs_axis, baseline_history['gradient_norm'], color='#6a4c93')
axes[1, 0].set(title='Mean mini-batch gradient norm', xlabel='epoch', ylabel='L2 norm', yscale='log')
plot_boundary(axes[1, 1], trained_parameters, 'tanh', 'Validation boundary after baseline training')
plt.tight_layout()
plt.show()

## Break It Safely: Prove the NaN Guard Stops

Predict the bounded message category before injecting one `nan` into a copied parameter dictionary. The canonical baseline remains untouched. Catching this deliberate failure demonstrates that a training run will stop at the first reported epoch/batch instead of continuing silently.

In [ ]:
nan_guard_prediction = ""
assert nan_guard_prediction.strip()
corrupted_parameters = {name: values.copy() for name, values in trained_parameters.items()}
corrupted_parameters['W1'][0, 0] = np.nan
nan_guard_observed = False
try:
    require_finite_state(corrupted_parameters, initial_gradients, epoch=7, batch_index=3)
except FloatingPointError as error:
    nan_guard_observed = True
    print("Expected bounded diagnostic:", error)
assert nan_guard_observed
assert all(np.all(np.isfinite(values)) for values in trained_parameters.values())

## Diagnose Two Different Failures

Commit before running: predict the boundary family and accuracy of an identity-hidden network, then predict the curve/gradient symptoms of learning rate `30.0`. Name one check that distinguishes representational limitation from unstable optimization. Both variants start from the same seed and use separate parameter dictionaries.

In [ ]:
failure_predictions = {
    "identity_boundary_and_accuracy": "",
    "high_rate_curve_and_gradients": "",
    "discriminating_check": "",
}
assert all(value.strip() for value in failure_predictions.values())

In [ ]:
identity_config = {**baseline_config, "activation": 'identity', "epochs": 300}
identity_parameters, identity_history = train_model(X_train, y_train, X_val, y_val, **identity_config)
identity_val_accuracy = classification_accuracy(y_val, forward(X_val, identity_parameters, 'identity')['P'])
high_rate_config = {**baseline_config, "learning_rate": 30.0, "epochs": 120}
high_rate_error = None
try:
    high_rate_parameters, high_rate_history = train_model(X_train, y_train, X_val, y_val, **high_rate_config)
except FloatingPointError as error:
    high_rate_parameters, high_rate_history, high_rate_error = None, None, str(error)
    print("High-rate run stopped by bounded diagnostic:", high_rate_error)
print(f"Identity-hidden validation accuracy: {identity_val_accuracy:.3f}")
if high_rate_history is not None:
    print(f"High-rate validation-loss range: {min(high_rate_history['val_loss']):.3f} to {max(high_rate_history['val_loss']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_boundary(axes[0], identity_parameters, 'identity', 'No activation: trained boundary')
axes[1].plot(identity_history['val_loss'], color='#277da1', label='identity validation loss')
axes[1].plot(baseline_history['val_loss'][:len(identity_history['val_loss'])], color='#f9844a', label='tanh validation loss')
axes[1].set(title='Representation comparison', xlabel='epoch', ylabel='BCE')
axes[1].legend()
if high_rate_history is None:
    axes[2].text(0.05, 0.6, high_rate_error, wrap=True)
    axes[2].set(title='High-rate bounded stop')
else:
    axes[2].plot(high_rate_history['val_loss'], color='#d1495b', label='validation loss')
    axes[2].plot(high_rate_history['gradient_norm'], color='#6a4c93', alpha=0.75, label='gradient norm')
    axes[2].set(title='Too-high learning-rate evidence', xlabel='epoch', yscale='log')
    axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
failure_diagnosis = {
    "identity_evidence_and_mechanism": "",
    "high_rate_evidence_and_mechanism": "",
    "targeted_recovery_for_each": "",
}
assert all(value.strip() for value in failure_diagnosis.values())

## Challenge: You Get One Isolated Experiment

Choose exactly one change: learning rate or hidden width. State a falsifiable hypothesis and the evidence expected in loss, accuracy, boundary, and gradient norm. Keep data, split, activation, batch size, epochs, and seeds fixed. Run once, then decide whether the evidence supports the hypothesis.

In [ ]:
experiment_change = ""  # TODO: 'learning_rate' or 'hidden_dim'.
experiment_value = None  # TODO: positive numeric value different from baseline.
experiment_hypothesis = ""
experiment_expected_evidence = {"loss": "", "accuracy": "", "boundary": "", "gradient_norm": ""}
assert experiment_change in {'learning_rate', 'hidden_dim'}
assert experiment_value is not None and float(experiment_value) > 0
assert not np.isclose(float(experiment_value), baseline_config[experiment_change])
assert experiment_hypothesis.strip() and all(value.strip() for value in experiment_expected_evidence.values())
experiment_config = baseline_config.copy()
experiment_config[experiment_change] = int(experiment_value) if experiment_change == 'hidden_dim' else float(experiment_value)
changed_keys = [key for key in baseline_config if experiment_config[key] != baseline_config[key]]
assert changed_keys == [experiment_change]

In [ ]:
experiment_parameters, experiment_history = train_model(X_train, y_train, X_val, y_val, **experiment_config)
experiment_val_cache = forward(X_val, experiment_parameters, experiment_config['activation'])
experiment_val_loss = binary_cross_entropy(y_val, experiment_val_cache['P'])
experiment_val_accuracy = classification_accuracy(y_val, experiment_val_cache['P'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(baseline_history['val_loss'], label='baseline', color='#277da1')
axes[0].plot(experiment_history['val_loss'], label='one-change run', color='#f9844a')
axes[0].set(title='Validation loss', xlabel='epoch', ylabel='BCE')
axes[0].legend()
axes[1].plot(baseline_history['gradient_norm'], label='baseline', color='#277da1')
axes[1].plot(experiment_history['gradient_norm'], label='one-change run', color='#f9844a')
axes[1].set(title='Gradient norm', xlabel='epoch', ylabel='L2 norm', yscale='log')
axes[1].legend()
plot_boundary(axes[2], experiment_parameters, experiment_config['activation'], f"One change: {experiment_change}={experiment_config[experiment_change]}")
plt.tight_layout()
plt.show()
print(f"Baseline validation loss/accuracy: {baseline_val_loss:.4f} / {baseline_val_accuracy:.3f}")
print(f"Experiment validation loss/accuracy: {experiment_val_loss:.4f} / {experiment_val_accuracy:.3f}")
experiment_interpretation = ""  # TODO: accept/reject the hypothesis using all four evidence types.
assert experiment_interpretation.strip()

## Optional Extension: Test the Other Variable

If the core challenge changed learning rate, optionally change hidden width; if it changed width, optionally change learning rate. Preserve every other baseline value and predict the evidence first. This extension is not required by the core checkpoint.

In [ ]:
optional_change = None  # TODO (optional): the variable not used in the core challenge.
optional_value = None
optional_prediction = ""
if optional_change is None or optional_value is None or not optional_prediction.strip():
    print("Optional second experiment skipped. Core checkpoint is unaffected.")
else:
    assert optional_change in {'learning_rate', 'hidden_dim'} and optional_change != experiment_change
    optional_config = baseline_config.copy()
    optional_config[optional_change] = int(optional_value) if optional_change == 'hidden_dim' else float(optional_value)
    assert [key for key in baseline_config if optional_config[key] != baseline_config[key]] == [optional_change]
    optional_parameters, optional_history = train_model(X_train, y_train, X_val, y_val, **optional_config)
    optional_val_cache = forward(X_val, optional_parameters, optional_config['activation'])
    print(f"Optional validation loss: {binary_cross_entropy(y_val, optional_val_cache['P']):.4f}")
    print(f"Optional validation accuracy: {classification_accuracy(y_val, optional_val_cache['P']):.3f}")
    optional_interpretation = ""  # TODO (optional): compare with the same baseline evidence.
    assert optional_interpretation.strip()

## Reflect and Checkpoint

Cite one piece of evidence for optimization progress, one for representational adequacy, and one for plausible generalization. Explain why decreasing loss alone cannot validate every stage, and map each implemented function to one training-loop responsibility.

In [ ]:
reflection = {
    "optimization_evidence": "",
    "representation_evidence": "",
    "generalization_evidence": "",
    "loss_alone_limit": "",
    "function_to_loop_mapping": "",
}
assert all(value.strip() for value in reflection.values())
assert 0.65 <= initial_loss <= 0.75
assert baseline_train_loss < 0.35 and baseline_val_loss < 0.35
assert 0.85 <= baseline_train_accuracy <= 0.95
assert 0.85 <= baseline_val_accuracy <= 0.95
assert baseline_val_accuracy >= identity_val_accuracy
assert nan_guard_observed
high_rate_is_unstable = high_rate_history is None or max(high_rate_history['val_loss']) > 1.0
assert high_rate_is_unstable
assert changed_keys == [experiment_change]
print("LAB-D2-03 checkpoint passed: stages, baseline bands, boundary/gradient evidence, bounded diagnostics, and one-change experiment.")

## Takeaways

- A complete training loop has separable responsibilities and testable invariants.
- Vectorization preserves the per-example function while adding a batch dimension.
- Stable loss and bounded finite-state checks turn silent corruption into actionable evidence.
- Loss, accuracy, boundary shape, and gradient norm test different hypotheses.
- One-variable experiments preserve attribution; a lower final scalar alone does not explain mechanism.

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Initial loss is far from the broad `0.65-0.75` band | Initialization scale, seed, or split changed | Restart and restore supplied constants |
| `P` has shape `(B,)` | Output dimension was squeezed | Preserve `(B, 1)` through forward and targets |
| Gradient shapes mismatch parameters | A transpose or batch reduction is wrong | Compare each matrix product with the shape map |
| One epoch misses or repeats examples | Batch iterator reshuffles inside the slice loop | Create one permutation per epoch |
| Non-finite diagnostic appears | Rate is unstable or clipping/gradient logic is wrong | Inspect the reported epoch/batch and lower the isolated rate |
| Loss falls but boundary stays linear | Hidden activation is identity or absent | Inspect activation choice and boundary evidence |
| Validation differs from the sanity band | Seed, standardization, or experiment state leaked | Restart and run the baseline before variants |

## Continue

Return to the [LAB-D2-03 debrief](../student-guide/day-2-student-guide.md#lab-d2-03---train-a-neural-network-with-numpy). Review [LESSON-D2-04](../student-guide/day-2-student-guide.md#lesson-d2-04---vectorization-tensors-and-mini-batches), [LESSON-D2-05](../student-guide/day-2-student-guide.md#lesson-d2-05---train-a-network-from-scratch), [ACT-D2-03](../challenges/day-2-challenges.md#act-d2-03---batch-shape-map), [LAB-D2-01](LAB-D2-01-loss-learning-rate.ipynb), and [LAB-D2-02](LAB-D2-02-backprop-gradient-check.ipynb) as needed.